# LRURec 项目实验记录
本文件来自实际执行的 Colab Notebook，已移除上传文件字节回显和前端控件。数据准备、运行方式和提交结果见 `docs/LRURec运行说明.md` 与 `docs/实验记录_lrurec.md`。
运行于 2026-11-17，固定官方模型版本；正式运行路径中的时间编号仅用作目录名。子进程训练日志未完整嵌入本 Notebook，请以逐轮验证记录和运行元数据核对结果。
从头复现可使用项目运行说明；本 Notebook 中的交互上传步骤供 Colab 使用。

In [1]:
from google.colab import files
files.upload()

In [2]:
from pathlib import Path
import zipfile
import os

ROOT = Path("/content/cs5481_project")
ROOT.mkdir(exist_ok=True)

with zipfile.ZipFile("/content/project.zip") as z:
    z.extractall(ROOT)

candidates = list(ROOT.glob("*/configs/config.yaml"))
assert len(candidates) == 1, "项目目录不唯一，请检查压缩包"

PROJECT = candidates[0].parent.parent
os.chdir(PROJECT)
print("项目目录：", PROJECT)

项目目录： /content/cs5481_project/cs5481-movie-recommendation-system-main


In [3]:
import subprocess
import sys
import urllib.request
import zipfile

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
    check=True,
)

archive = Path("/content/ml-1m.zip")
urllib.request.urlretrieve(
    "https://files.grouplens.org/datasets/movielens/ml-1m.zip",
    archive,
)

with zipfile.ZipFile(archive) as z:
    z.extractall(PROJECT)

subprocess.run(
    [sys.executable, "src/preprocess.py"],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', 'src/preprocess.py'], returncode=0)

In [4]:
import pandas as pd
import yaml

with open("configs/config.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

processed = Path(cfg["paths"]["processed_dir"])
threshold = cfg["labels"]["positive_threshold"]

train = pd.read_csv(processed / "train.csv")
val = pd.read_csv(processed / "val.csv")
test = pd.read_csv(processed / "test.csv")

required = {"UserID", "MovieID", "Rating", "Timestamp", "label"}

for name, df in [("train", train), ("val", val), ("test", test)]:
    assert required <= set(df.columns), f"{name} 缺少必要字段"
    assert not df[list(required)].isna().any().any(), f"{name} 有缺失值"
    assert not df.duplicated(["UserID", "MovieID"]).any(), f"{name} 有重复评分"
    assert (df["label"] == (df["Rating"] >= threshold).astype(int)).all()

for name, df in [("val", val), ("test", test)]:
    assert (df["label"] == 1).all(), f"{name} 包含非正例"
    assert df.groupby("UserID").size().le(1).all()

combined = pd.concat([
    train.assign(split="train"),
    val.assign(split="val"),
    test.assign(split="test"),
])
assert (
    combined.groupby(["UserID", "MovieID"])["split"].nunique() <= 1
).all(), "同一用户电影对跨集合重复"

train_pos = train[train["label"] == 1].copy()
positive_counts = train_pos.groupby("UserID").size()

# 至少两个训练正例，才能构成一组历史 -> 下一部电影训练样本
trainable_users = set(positive_counts[positive_counts >= 2].index)
train_history_users = set(positive_counts.index)

# 有至少一个训练正例历史，可参与验证
val_users = train_history_users & set(val["UserID"])

# 有训练历史和真实验证记录，参与标准测试
test_users = (
    train_history_users
    & set(val["UserID"])
    & set(test["UserID"])
)

summary = pd.DataFrame([
    {"项目": "train 全部评分", "数量": len(train)},
    {"项目": "train 正例", "数量": len(train_pos)},
    {"项目": "具备至少两个训练正例的用户", "数量": len(trainable_users)},
    {"项目": "验证有效用户", "数量": len(val_users)},
    {"项目": "测试有效用户", "数量": len(test_users)},
    {"项目": "有测试目标但不满足上述测试条件的用户",
     "数量": len(set(test["UserID"]) - test_users)},
])

display(summary)

# 时间戳相同的记录需要统一排序规则，因此这里只检查严格晚于目标的情况
last_train_positive = train_pos.groupby("UserID")["Timestamp"].max()
val_time = val.set_index("UserID")["Timestamp"]
test_time = test.set_index("UserID")["Timestamp"]

common_val = sorted(val_users)
common_test = sorted(test_users)

assert (
    last_train_positive.reindex(common_val)
    <= val_time.reindex(common_val)
).all(), "存在验证目标之后的训练正例"

assert (
    val_time.reindex(common_test)
    <= test_time.reindex(common_test)
).all(), "存在测试目标之后的验证记录"

print("数据基础检查通过。未伪造验证答案，未使用低分记录补训练序列。")

                   项目      数量
0          train 全部评分  988134
1            train 正例  563206
2       具备至少两个训练正例的用户    6035
3              验证有效用户    6035
4              测试有效用户    6035
5  有测试目标但不满足上述测试条件的用户       3

数据基础检查通过。未伪造验证答案，未使用低分记录补训练序列。


In [5]:
import platform
import subprocess
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("当前没有 GPU；先切换 GPU 运行环境，再开始正式训练。")

subprocess.run(
    ["git", "clone", "https://github.com/yueqirex/LRURec.git",
     "/content/LRURec_project"],
    check=True,
)

print("LRURec 代码版本：")
subprocess.run(
    ["git", "-C", "/content/LRURec_project", "rev-parse", "HEAD"],
    check=True,
)

Python: 3.13.16
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
LRURec 代码版本：


CompletedProcess(args=['git', '-C', '/content/LRURec_project', 'rev-parse', 'HEAD'], returncode=0)

In [6]:
from google.colab import files
files.upload()

In [8]:
from pathlib import Path
import subprocess, sys, shutil

candidates = list(Path('/content/cs5481_project').glob('*/configs/config.yaml'))
assert len(candidates) == 1, '请先运行准备小组项目和数据的单元'
PROJECT = candidates[0].parent.parent
RUNNER = PROJECT / 'src/lrurec_project.py'

from google.colab import files

uploaded = files.upload()

matches = [
    name for name in uploaded
    if name.endswith(".py") and "lrurec_project" in name
]
assert len(matches) == 1, "请选择一个 lrurec_project.py 文件"

RUNNER.parent.mkdir(parents=True, exist_ok=True)
RUNNER.write_bytes(uploaded[matches[0]])

print("脚本已保存到：", RUNNER)

subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyyaml'], check=True)

OFFICIAL = Path('/content/LRURec_project')
PIN = 'e02a559bd8c42342f5c53d69cefea2589becb22c'
if not OFFICIAL.exists():
    subprocess.run(['git','clone','https://github.com/yueqirex/LRURec.git',str(OFFICIAL)],check=True)
dirty = subprocess.check_output(['git','-C',str(OFFICIAL),'status','--porcelain'],text=True)
assert not dirty.strip(), '官方目录存在修改，请使用一个新的官方代码目录'
subprocess.run(['git','-C',str(OFFICIAL),'checkout','--detach',PIN],check=True)
print('官方提交编号：', subprocess.check_output(['git','-C',str(OFFICIAL),'rev-parse','HEAD'],text=True).strip())

import torch
assert torch.cuda.is_available(), '请启用 GPU'
print('PyTorch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))


<IPython.core.display.HTML object>

Saving lrurec_project.py to lrurec_project (1).py
脚本已保存到： /content/cs5481_project/cs5481-movie-recommendation-system-main/src/lrurec_project.py
官方提交编号： e02a559bd8c42342f5c53d69cefea2589becb22c
PyTorch: 2.14.1+cu130 GPU: Tesla T4


In [9]:
import time
SMOKE = Path('/content') / ('lrurec_smoke_' + str(time.time_ns()))
subprocess.run([sys.executable, '-u', str(RUNNER), '--project', str(PROJECT),
    '--official', str(OFFICIAL), '--output', str(SMOKE),
    '--epochs','2','--smoke'],check=True)


CompletedProcess(args=['/usr/bin/python3', '-u', '/content/cs5481_project/cs5481-movie-recommendation-system-main/src/lrurec_project.py', '--project', '/content/cs5481_project/cs5481-movie-recommendation-system-main', '--official', '/content/LRURec_project', '--output', '/content/lrurec_smoke_1794930462986940634', '--epochs', '2', '--smoke'], returncode=0)

In [11]:
from pathlib import Path
import time
import subprocess
import sys

RUN = Path("/content/CS5481_LRURec") / ("run_" + str(time.time_ns()))

subprocess.run([
    sys.executable, "-u", str(RUNNER),
    "--project", str(PROJECT),
    "--official", str(OFFICIAL),
    "--output", str(RUN),
    "--epochs", "50",
], check=True)

print("本次输出：", RUN)

本次输出： /content/CS5481_LRURec/run_1794930764979407342


In [12]:
import pandas as pd, json, zipfile
metadata = json.loads((RUN/'run_metadata.json').read_text())
assert metadata['status'] == 'completed'
display(pd.read_csv(RUN/'10_lrurec_metrics.csv'))
display(pd.read_csv(RUN/'12_lrurec_final_top5.csv'))
print('评估用户范围：', metadata['population'])
print('最佳轮次：', metadata['best_epoch'])
print('模型参数量：', metadata['model_parameters'])


    Model  Seed  Best_Epoch  N_Test_Users  Recall@1    NDCG@1     MRR@1  \
0  LRURec    42          38          6035  0.045568  0.045568  0.045568   

   Recall@5    NDCG@5     MRR@5  Recall@10   NDCG@10    MRR@10  Recall@20  \
0  0.152444  0.098991  0.081541   0.243413  0.128483  0.093756   0.361723   

    NDCG@20    MRR@20  Recall@50   NDCG@50    MRR@50  
0  0.158193  0.101806   0.531069  0.191904  0.107263  

    UserID  Rank  MovieID                                       Title  \
0        1     1     1688                            Anastasia (1997)   
1        1     2     2394                 Prince of Egypt, The (1998)   
2        1     3      364                       Lion King, The (1994)   
3        1     4     1033               Fox and the Hound, The (1981)   
4        1     5     2083          Muppet Christmas Carol, The (1992)   
5        2     1     1586                            G.I. Jane (1997)   
6        2     2     2023             Godfather: Part III, The (1990)   
7        2     3     2763             Thomas Crown Affair, The (1999)   
8        2     4     1479                           Saint, The (1997)   
9        2     5      208                           Waterworld (1995)   
10       3     1     2797                                  Big (1988)   
11       3     2     2918             Ferris Bueller's Day Off (1986)   
12       3     3     2716                         G

评估用户范围： {'train_users': 6035, 'val_users': 6035, 'test_users': 6035, 'n_items': 3706}
最佳轮次： 38
模型参数量： 341691


In [13]:
from pathlib import Path
import zipfile
import shutil
from google.colab import files

RUN = Path("/content/CS5481_LRURec/run_1794930764979407342")
assert RUN.is_dir(), f"找不到实验目录：{RUN}"

# 1. 复核包：脚本、指标和实验记录，不含模型权重
archive = Path("/content/lrurec_review_bundle.zip")

with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(RUNNER, "src/lrurec_project.py")
    for p in RUN.iterdir():
        if p.is_file() and p.suffix in (".csv", ".json"):
            z.write(p, "experiment/" + p.name)

files.download(str(archive))

# 2. 完整备份：包含 best_model.pth
backup = shutil.make_archive(
    "/content/lrurec_full_backup",
    "zip",
    root_dir=str(RUN.parent),
    base_dir=RUN.name,
)
files.download(backup)